In [ ]:
# === Core Python ===
import os
import glob
from typing import Tuple, Dict, Optional

# === Numerical & Data Handling ===
import numpy as np
import pandas as pd
import xarray as xr
import xcdat as xcd

# === Plotting & Visualization ===
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# === Evaluation Metrics ===
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


def _to_datetimeindex(index):
    try:
        return index.to_datetimeindex(time_unit="ns")
    except TypeError:
        return index.to_datetimeindex()


In [ ]:
class ObsDataReader:
    """
    Class to read and preprocess observational data with internal dataset registry.
    Supports regional mean extraction.
    """

    def __init__(self):
        self.obs_group = self._define_obs_group()

    def _define_obs_group(self):
        return {
            'ERA5': {
                'monthly': {
                    'path': '/compyfs/zhan391/acme_init/Observations/ERA5/monthly',
                    'template': 'ERA5_analysis_monthly_%(year).nc',
                    'period': '2001-2018'
                },
                '6hourly': {
                    'path': '/compyfs/zhan391/acme_init/Observations/ERA5/6hourly',
                    'template': 'ERA5.6hourly.en00.%(var).%(year)01-%(year)12.nc',
                    'period': '1979-2018'
                }
            },
            'GPCP': {
                'monthly': {
                    'path': '/compyfs/zhan391/acme_init/Observations/GPCP/monthly',
                    'template': 'PRECT.monthly.%(year).nc',
                    'period': '1979-2017'
                },
                'daily': {
                    'path': '/compyfs/zhan391/acme_init/Observations/GPCP/daily',
                    'template': 'PRECT.daily.%(year).nc',
                    'period': '2007-2011'
                }
            },
            'IMERG': {
                'daily': {
                    'path': '/compyfs/zhan391/acme_init/Observations/IMERG/daily',
                    'template': 'PRECT.daily.%(year).nc',
                    'period': '2007-2011'
                }
            }
        }

    @staticmethod
    def define_region(regnam='global'):
        reg_dict = {
            'global':     [(-90, 90), (-180, 180)],
            'NHMidLat':   [(25, 50), (-60, 150)],
            'Tropics':    [(-10, 10), (-90, 60)],
            'Atlantic':   [(5, 55), (-95, -40)],
            'CONUS':      [(25, 50), (-125, -95)],
            'Antarctic':  [(-90, -50), (-180, 180)],
            'PolarN':     [(50, 90), (-180, 180)],
            'Greenland':  [(60, 85), (-75, -10)],
        }
        if regnam not in reg_dict:
            raise ValueError(f"Region '{regnam}' not defined. Available: {list(reg_dict.keys())}")
        return reg_dict[regnam]

    def get_group(self, name=None):
        return self.obs_group.get(name) if name else self.obs_group

    def list_datasets(self):
        return list(self.obs_group.keys())

    def _ensure_datetime64(self, da: xr.DataArray) -> xr.DataArray:
        """Convert time to datetime64 if cftime is used."""
        if not isinstance(da.indexes['time'], pd.DatetimeIndex):
            da['time'] = da.indexes['time'].to_datetimeindex(time_unit='ns')
        return da

    def read(self, var, obsname, time, frequency=None, regnam='global', regional_mean=False):
        obs_group = self.obs_group.get(obsname)
        if obs_group is None:
            raise ValueError(f"Observational dataset '{obsname}' not found.")
        if frequency not in obs_group:
            raise ValueError(f"Frequency '{frequency}' not found in '{obsname}'")

        group_info = obs_group[frequency]
        time_index = pd.to_datetime(time)
        years = sorted(set(str(y) for y in time_index.year))
        
        paths = []
        for y in years:
            fname = group_info["template"].replace("%(year)", y).replace("%(var)", var)
            paths.append(os.path.join(group_info["path"], fname))

        refds = xr.open_mfdataset(paths, decode_times=True, combine='by_coords')
        if refds.lon.min() >= 0:
            refds = refds.assign_coords(lon=((refds.lon + 180) % 360 - 180)).sortby("lon")

        if var not in refds:
            raise KeyError(f"Variable '{var}' not found in file(s): {paths}")

        data = self.convert_obs_units(var, refds[var])
        data = self._ensure_datetime64(data)

        (lat1, lat2), (lon1, lon2) = self.define_region(regnam)
        data = data.sel(lat=slice(lat1, lat2), lon=slice(lon1, lon2))

        # Nearest time match within 1-day tolerance
        data = data.sel(time=time, method='nearest', tolerance=np.timedelta64(1, 'D'))

        if regional_mean:
            weights = np.cos(np.deg2rad(data.lat))
            weights /= weights.mean()
            data = (data * weights).mean(dim=["lat", "lon"])

        return data.chunk({"time": -1}) if 'time' in data.dims else data

    def convert_obs_units(self, var, data_array):
        conversions = {
            'PRECT': lambda x: x * 86400.0 * 1000.0,
            'TREFHT': lambda x: x - 273.15,
            'TS': lambda x: x - 273.15,
            'T850': lambda x: x - 273.15,
            'PSL': lambda x: x / 100.0,
            'PS': lambda x: x / 100.0,
        }

        if 'time' not in data_array.dims:
            raise ValueError(f"Expected 'time' dimension in data for '{var}'")

        return conversions[var](data_array) if var in conversions else data_array


In [ ]:
class ModelDataReader:
    def __init__(self, base_path, exp_base, regnam, frequency, component="atm"):
        self.base_path = base_path
        self.exp_base = exp_base
        self.regnam = regnam
        self.frequency = frequency
        self.component = component
        self.region = self.define_region(regnam)
        self.exp_dict = self.extract_exp_info(exp_base, base_path)

    def _ensure_datetime64(self, da: xr.DataArray) -> xr.DataArray:
        """Convert cftime.datetime to datetime64[ns] if needed."""
        if not isinstance(da.indexes['time'], pd.DatetimeIndex):
            da['time'] = da.indexes['time'].to_datetimeindex(time_unit='ns')
        return da

    @staticmethod
    def define_region(regnam: str = 'global') -> Tuple[Tuple[float, float], Tuple[float, float]]:
        reg_dict = {
            'global':     [(-90, 90), (-180, 180)],
            'NHMidLat':   [(25, 50), (-60, 150)],
            'Tropics':    [(-10, 10), (-90, 60)],
            'Atlantic':   [(5, 55), (-95, -40)],
            'CONUS':      [(25, 50), (-125, -95)],
            'Antarctic':  [(-90, -50), (-180, 180)],
            'PolarN':     [(50, 90), (-180, 180)],
            'Greenland':  [(60, 85), (-75, -10)],
        }
        if regnam not in reg_dict:
            raise ValueError(f"Region '{regnam}' not defined. Available: {list(reg_dict.keys())}")
        return reg_dict[regnam]

    @staticmethod
    def extract_exp_info(base_name, data_path) -> dict:
        exps = {
            'CTRLEN10':       {'name': 'ctrl_en10', 'nens': 10, 'period': '201112'},
            'DARTEN10':       {'name': 'dart_en10', 'nens': 10, 'period': '201112'},
            'DARTEN20':       {'name': 'dart_en20', 'nens': 20, 'period': '201112'},
            'DARTEN40':       {'name': 'dart_en40', 'nens': 40, 'period': '201112'},
            'CAPTEN10_15day': {'name': 'capt_en10', 'nens': 10, 'period': '201201-201202'},
            'DARTEN20_15day': {'name': 'dart_en20', 'nens': 20, 'period': '201201-201202'},
            'DARTEN40_15day': {'name': 'dart_en40', 'nens': 40, 'period': '201201-201202'},
        }
        return {
            exp_name: {
                'run': f"{exp_name}_{base_name}" if base_name else exp_name,
                'key': exp_data['name'],
                'nens': exp_data['nens'],
                'atm': 'archive/post/atm/180x360_aave',
                'lnd': 'archive/post/lnd/180x360_aave',
                'period': exp_data['period']
            }
            for exp_name, exp_data in sorted(exps.items())
        }

    def _convert_model_units(self, var, data):
        if var == 'PRECT':
            return data * 86400.0 * 1000.0  # m/s to mm/day
        elif var in ['TREFHT', 'TS', 'T850']:
            return data - 273.15  # K to C
        elif var in ['PSL', 'PS']:
            return data / 100.0  # Pa to hPa
        return data

    def _read_variable_single_experiment(self, var: str, model_name: str, **kwargs) -> xr.DataArray:
        return self.read_variable(var, model_name, **kwargs)

    def read_variable(self, var, exp, frequency=None, regional_mean=False):
        """
        Load and process a model variable, optionally computing regional mean.
        """
        run = self.exp_dict[exp]['run']
        component_path = self.exp_dict[exp][self.component]
        nens = self.exp_dict[exp]['nens']
        period_raw = self.exp_dict[exp]['period']
        freq = frequency or self.frequency

        if freq in ['6hourly', 'daily']:
            prefix = 'ts/daily'
            period = period_raw.split("-")[0][:4]
        elif freq == 'monthly':
            prefix = 'clim/monthly'
            period = period_raw.split("-")[0][:4] + "*"
        else:
            raise ValueError(f"Unsupported frequency: {freq}")

        (lat1, lat2), (lon1, lon2) = self.region
        ensemble_data = []

        for i in range(1, nens + 1):
            member = f"EN{i:02d}"
            search_dir = os.path.join(self.base_path, run, component_path, prefix)
            file_pattern = f"{var}.{member}.{period}.nc"
            search_path = os.path.join(search_dir, file_pattern)
            rpath = sorted(glob.glob(search_path))

            if not rpath:
                raise FileNotFoundError(f"No files for {member}: {search_path}")

            ds = xcd.open_mfdataset(
                rpath,
                combine='nested',
                concat_dim='time',
                coords='minimal',
                compat='override'
            )

            if ds.lon.min() >= 0:
                ds = ds.assign_coords(lon=((ds.lon + 180) % 360 - 180)).sortby("lon")

            if var not in ds:
                raise KeyError(f"Variable '{var}' missing in {rpath[0]}")

            data = self._convert_model_units(var, ds[var])
            data = data.sel(lat=slice(lat1, lat2), lon=slice(lon1, lon2))

            if regional_mean:
                weights = np.cos(np.deg2rad(data.lat))
                weights /= weights.mean()
                data = (data * weights).mean(dim=["lat", "lon"])

            data = self._ensure_datetime64(data)
            ensemble_data.append(data.expand_dims(member=[i]))

        combined = xr.concat(ensemble_data, dim="member")

        if regional_mean:
            return combined.chunk({"member": -1, "time": -1})
        else:
            return combined.chunk({"member": -1, "time": -1, "lat": -1, "lon": -1})

    def read_variable_across_experiments(self, var: str, model_list: list, **kwargs) -> Dict[str, xr.DataArray]:
        data_dict = {}
        for model_name in model_list:
            da = self._read_variable_single_experiment(var, model_name, **kwargs)
            data_dict[model_name] = da
        return data_dict


In [ ]:
class TimeSeriesComparisonPlotter:
    """
    Plot time series from multiple experiments and a reference dataset.
    Supports ensemble visualization:
    - Mean line
    - Min/max shading
    - ±1σ whisker bars with horizontal caps (boxplot style)
    """

    def __init__(self, model_data: Dict[str, xr.DataArray], obs_data: Optional[xr.DataArray],
                 output_dir: str, var: Dict, units: Dict, region: str, regnam: str):
        self.model_data = model_data
        self.obs_data = obs_data
        self.output_dir = output_dir
        self.var = list(var.keys())[0]
        self.units = units[self.var]['unit']
        self.region = region
        self.regnam = regnam

    def _ensure_datetime64(self, da: xr.DataArray):
        """Convert cftime to datetime64 if needed."""
        if not isinstance(da.indexes['time'], pd.DatetimeIndex):
            da['time'] = da.indexes['time'].to_datetimeindex(time_unit='ns')  # suppress FutureWarning
        return da

    def _compute_metrics(self, model: xr.DataArray, obs: xr.DataArray):
        from sklearn.metrics import mean_squared_error
        from scipy.stats import pearsonr

        obs = obs.sel(time=model.time)

        ensemble_dim = next((d for d in model.dims if d != 'time'), None)
        rmse_list = []
        corr_list = []

        if ensemble_dim:
            for i in range(model.sizes[ensemble_dim]):
                member = model.isel({ensemble_dim: i})
                if np.all(np.isnan(member.values)):
                    continue
                rmse = np.sqrt(mean_squared_error(obs.values, member.values))
                corr, _ = pearsonr(obs.values, member.values)
                rmse_list.append(rmse)
                corr_list.append(corr)
            rmse_mean = np.mean(rmse_list)
            rmse_std = np.std(rmse_list)
            corr_mean = np.mean(corr_list)
            corr_std = np.std(corr_list)
        else:
            rmse_mean = np.sqrt(mean_squared_error(obs.values, model.values))
            corr_mean, _ = pearsonr(obs.values, model.values)
            rmse_std = 0
            corr_std = 0

        return rmse_mean, rmse_std, corr_mean, corr_std
    
    def plot(self, figsize=(12, 6), linewidth=2, alpha=0.3, save=True, fontz=14, mean_metrics_on=False):
        plt.figure(figsize=figsize)
        sample_color = None
        legend_elements = []

        # Determine global reference time
        all_times = []
        for da in self.model_data.values():
            da = self._ensure_datetime64(da)
            all_times.append(da.time.values)
        if self.obs_data is not None:
            self.obs_data = self._ensure_datetime64(self.obs_data)
            all_times.append(self.obs_data.time.values)

        all_times_flat = np.concatenate(all_times)
        reference_time = pd.to_datetime(all_times_flat.min())
        ref_label = reference_time.strftime('%Y-%m-%d')

        def days_since(t):
            return (pd.to_datetime(t) - reference_time).days
        
        # Collect metrics strings to display in a text box
        metrics_text_lines = []

        for i, (exp_name, da) in enumerate(self.model_data.items()):
            da = self._ensure_datetime64(da)
            color = cm.tab10(i % 10)
            if sample_color is None:
                sample_color = color

            print(f"[INFO] {exp_name} dims: {da.dims}")

            ensemble_dim = next((dim for dim in ['ens', 'member'] if dim in da.dims), None)

            if ensemble_dim:
                if da.dims[0] != 'time':
                    da = da.transpose('time', ensemble_dim)
                time_days = np.array([days_since(t) for t in da.time.values])
                mean = da.mean(dim=ensemble_dim)
                std = da.std(dim=ensemble_dim)
                min_ = da.min(dim=ensemble_dim)
                max_ = da.max(dim=ensemble_dim)

                plt.plot(time_days, mean, label=None, color=color, linewidth=linewidth)
                plt.vlines(time_days, mean - std, mean + std, color=color, alpha=0.6, linewidth=1.5)

                if len(time_days) > 1:
                    delta = (time_days[1] - time_days[0]) * 0.25
                else:
                    delta = 1

                for t, m, s in zip(time_days, mean.values, std.values):
                    plt.hlines(m - s, t - delta, t + delta, color=color, linewidth=1)
                    plt.hlines(m + s, t - delta, t + delta, color=color, linewidth=1)

                plt.fill_between(time_days, min_, max_, color=color, alpha=0.15)
            else:
                other_dims = [d for d in da.dims if d != 'time']
                if other_dims:
                    print(f"[WARN] {exp_name} has extra dims {da.dims}; reducing to mean.")
                    da = da.mean(dim=other_dims, skipna=True)
                time_days = np.array([days_since(t) for t in da.time.values])
                plt.plot(time_days, da.values, label=None, color=color, linewidth=linewidth)

            # === Metric Computation ===
            if self.obs_data is not None and mean_metrics_on:
                obs = self._ensure_datetime64(self.obs_data).interp(time=da.time)
                rmse_vals = []
                corr_vals = []
                for mem in da[ensemble_dim].values:
                    member_ts = da.sel({ensemble_dim: mem})
                    rmse_val = rmse(member_ts, obs, dim='time', skipna=True)
                    corr_val = pearson_r(member_ts, obs, dim='time', skipna=True)
                    rmse_vals.append(rmse_val.values)
                    corr_vals.append(corr_val.values)
                rmse_mean = np.nanmean(rmse_vals)
                rmse_std = np.nanstd(rmse_vals)
                corr_mean = np.nanmean(corr_vals)
                corr_std = np.nanstd(corr_vals)
            
                metrics_text_lines.append(
                    f"{exp_name}:\n  RMSE = {rmse_mean:.2f} ± {rmse_std:.2f}\n  CORR = {corr_mean:.2f} ± {corr_std:.2f}"
                )
        
            legend_elements.append(Line2D([0], [0], color=color, lw=linewidth, label=exp_name))

        if self.obs_data is not None:
            time_days = np.array([days_since(t) for t in self.obs_data.time.values])
            plt.plot(time_days, self.obs_data.values, label=None, color='black', linestyle='--', linewidth=2, marker='o')
            legend_elements.append(Line2D([0], [0], color='black', linestyle='--', linewidth=2,
                                          marker='o', label='Observation'))

        legend_elements.append(Line2D(
            [0], [0], color='gray', lw=linewidth,
            marker='|', markersize=10, markeredgewidth=1.5,
            label='±1σ Whiskers + Mean'
        ))

        legend_elements.append(Patch(facecolor='gray', alpha=0.15, label='Min/Max Range'))

        # Add metrics text box
        if metrics_text_lines and mean_metrics_on:
            metrics_text = "\n\n".join(metrics_text_lines)
            plt.gcf().text(
                0.73, 0.25, metrics_text,
                fontsize=fontz - 2, va='top', ha='left',
                bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.5')
            )
    
        # Labels and layout
        plt.title(f"{self.var} Time Series over {self.region}", fontsize=fontz + 2)
        plt.ylabel(f"{self.var} [{self.units}]", fontsize=fontz)
        plt.xlabel(f"Days since {ref_label}", fontsize=fontz)
        plt.xticks(fontsize=fontz - 2)
        plt.yticks(fontsize=fontz - 2)
        plt.grid(True, linewidth=0.6)
        plt.legend(handles=legend_elements, loc='upper right', fontsize=fontz - 2)
        plt.tight_layout()
        plt.show()

        if save:
            os.makedirs(self.output_dir, exist_ok=True)
            fname = os.path.join(self.output_dir, f"timeseries_{self.var}_{self.regnam}.pdf")
            plt.savefig(fname, dpi=600)
            print(f"Saved plot to: {fname}")

        plt.close()


In [ ]:
if __name__ == "__main__":
    top_path = "/pscratch/sd/z/zhan391/e3sm_dart"
    out_path = os.path.join(top_path, "diag_dart")
    summary_dir = os.path.join(out_path, "summary_metrics_netcdf")
    fig_path = "/global/homes/z/zhan391/analysis/diagnostic/figures"

    # === Setup ===
    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    obsname = "IMERG"
    var_dict = {"PRECT": {"unit": "mm/day", "alias": "PRECT", "min": 0, "max": 12}}
    var = list(var_dict.keys())[0]
    frequency = "daily"

    model_list = ["CTRLEN10_15day", "CAPTEN10_15day", "DARTEN20_15day", "DARTEN40_15day"]

    # === Region aliases for figure titles ===
    regnam_dict = {
        "global": "Global",
        "CONUS": "Continental US",
        "NHMidLat": "Mid-Latitude (NH)",
        "Tropics": "Tropics",
    }

    for regnam, region_alias in regnam_dict.items():
        print(f"\n=== Processing region: {regnam} ===")

        # === Read model data ===
        reader = ModelDataReader(
            base_path=top_path,
            exp_base=exp_base,
            regnam=regnam,
            frequency=frequency,
            component="atm",
        )
        model_data = reader.read_variable_across_experiments(
            var, model_list, frequency=frequency, regional_mean=True
        )

        # === Determine common model time range ===
        start = max(pd.to_datetime(da.time.min().item()) for da in model_data.values())
        end = min(pd.to_datetime(da.time.max().item()) for da in model_data.values())
        freq_str = "D" if frequency == "daily" else "6H"
        common_time = pd.date_range(start=start, end=end, freq=freq_str)

        # === Align model data ===
        for exp in model_data:
            model_data[exp] = model_data[exp].sel(time=common_time)

        # === Read observation data and align to model time ===
        obs_reader = ObsDataReader()
        obs_data = obs_reader.read(
            var=var,
            obsname=obsname,
            time=common_time,
            frequency=frequency,
            regnam=regnam,
            regional_mean=True,
        ).sel(time=common_time)

        # === Plot ===
        plotter = TimeSeriesComparisonPlotter(
            model_data=model_data,
            obs_data=obs_data,
            output_dir=fig_path,
            var=var_dict,
            units=var_dict,
            regnam=regnam,
            region=region_alias,
        )
        plotter.plot(fontz=16, mean_metrics_on=True)
